# v03 · SHA-256 label + base-N payload
`PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster-hash.py`. The 64-character address becomes a hash; regeneration needs the payload.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
mod = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster-hash.py", "hh")
HW = mod.HyperWebster()
rec = HW.index_text("Hello, World!")
print(rec.label, len(rec.label))
print(repr(rec.payload), rec.length)
assert HW.regenerate_from_record(rec) == "Hello, World!"

949789b2d2ec3c377c5989ba0b283238efcf9250f693a5c971eedc5183212146 64
'Jr;;p.~Ept;f!' 13


## Label length is fixed; the payload is the address and grows like it

In [3]:
print("chars  label  payload_chars  payload_bits(8/char)  raw_bits  payload/raw")
for size in (140, 1000, 4000):
    r = HW.index_text(CORPUS[:size])
    assert HW.regenerate_from_record(r) == CORPUS[:size]
    pb = 8*len(r.payload)
    print("%5d  %5d  %13d  %20d  %8d  %.3f" % (size, len(r.label), len(r.payload), pb, 8*size, pb/(8*size)))

chars  label  payload_chars  payload_bits(8/char)  raw_bits  payload/raw
  140     64            140                  1120      1120  1.000
 1000     64           1000                  8000      8000  1.000
 4000     64           4000                 32000     32000  1.000


## The label alone cannot regenerate anything
It is SHA-256 of the integer address. Regeneration is the payload path; the label is a lookup key. (Payload chars use the keyboard-order charset, one digit per character, so `payload/raw` ≈ 1.0 at 8 bits per stored character; measured above.)

In [4]:
r = HW.index_text("abc")
try:
    HW.regenerate(r.label, r.length)
except Exception as e:
    print(type(e).__name__, "-", str(e)[:80])
# with the label's own length supplied, the label decodes without complaint to an unrelated string
g = HW.regenerate(r.label, len(r.label))
print("label decoded as a payload ->", len(g), "chars, equals 'abc':", g == "abc")

ValueError - Length mismatch: got 64, expected 3. Payload or length may be corrupted.
label decoded as a payload -> 64 chars, equals 'abc': False
